# Stage 10 - RL self-play with batched MCTS

Branch `perf/batched-mcts` of `TahaSalmani/Chess`.

**Runtime -> Change runtime type -> Hardware accelerator: GPU (T4).**

What runs below:
1. environment + GPU check
2. split benchmark: per simulation, how much is python tree work vs neural-net forward
3. correctness benchmark: batched search must give the same result as searching each root alone
4. full `stage_10_Pytorch_RL.py` (self-play + RL training step)

`POOL_SIZE` in `params.yaml` controls how many games are searched in one batched forward. Bigger pool = better GPU use, until the python tree work becomes the bottleneck.

In [ ]:
import os, subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout.strip())
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
cores = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else os.cpu_count()
print('usable vCPU:', cores)

In [ ]:
!rm -rf /content/Chess
!git clone --branch perf/batched-mcts --single-branch --depth 1 https://github.com/TahaSalmani/Chess.git /content/Chess
%cd /content/Chess
!pip install -q python-chess zstandard scikit-learn tensorboard python-box==6.0.2 ensure pyyaml
!pip install -q -e .
!git log --oneline -1

## Optional: bring trained weights

Stage 10 loads `artifacts/prepare_torch_callbacks/trained_model.pth` if it exists. On a fresh Colab session it does not, so the network starts from random weights.

Random weights do **not** change the timing numbers - the search does the same amount of work - but the games themselves will be meaningless. Run the cell below only if you keep a checkpoint in Google Drive.

In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

SRC = '/content/drive/MyDrive/chess/trained_model.pth'   # <-- edit
DST = '/content/Chess/artifacts/prepare_torch_callbacks/trained_model.pth'
if os.path.exists(SRC):
    os.makedirs(os.path.dirname(DST), exist_ok=True)
    shutil.copy2(SRC, DST)
    print('copied', SRC, '->', DST)
else:
    print('not found:', SRC, '| continuing with random weights')

## 1. Where the time goes

`stub` = a network that returns zeros instantly, so that column is pure python tree work. `nn` = the difference, i.e. the forward passes.

On a GPU the `nn` column stays almost flat as the pool grows, while the `python` column grows linearly. The pool size where the two cross is your optimum.

In [ ]:
!python research/bench_mcts_split.py

## 2. Correctness + speedup

Checks that searching N roots in one batch gives the same visit counts as searching each root alone, that duplicate roots stay identical (no cross-talk between games), that priors only ever land on legal moves, and that the self-play tensors have the right shape. Then it sweeps pool sizes over equal total work.

In [ ]:
!python research/bench_mcts.py

## 3. Full stage 10

Uses `params.yaml` as-is: `NUM_SIMULATION`, `NUM_GAMES`, `MAX_MOVES`, `POOL_SIZE`. Progress is logged every 10 move-rounds with elapsed time and ETA.

In [ ]:
!cat params.yaml
!python src/Chess/pipeline/stage_10_Pytorch_RL.py 2>&1 | tee stage10.log

In [ ]:
!ls -la artifacts/rl_model artifacts/self_play_data
from google.colab import files
files.download('/content/Chess/artifacts/rl_model/rl_trained_model.pth')